# Practical 04 — Polynomial Regression

Observe how polynomial degree changes training/test error (bias–variance trade-off).

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


## 2. Load dataset and select one predictor

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame.sample(n=4000, random_state=42)
X = df[["MedInc"]]
y = df["MedHouseVal"]


## 3. Split data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 4. Fit models with different polynomial degrees

In [ ]:
degrees = [1, 2, 3, 5, 8]
models, rows = {}, []
for degree in degrees:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        LinearRegression()
    )
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    models[degree] = model
    rows.append({
        "Degree": degree,
        "Train MSE": mean_squared_error(y_train, train_pred),
        "Test MSE": mean_squared_error(y_test, test_pred),
        "Test R2": r2_score(y_test, test_pred)
    })
results = pd.DataFrame(rows)
results


## 5. Visualize fitted curves

In [ ]:
x_line = np.linspace(X["MedInc"].min(), X["MedInc"].max(), 300).reshape(-1, 1)
plt.scatter(X_test["MedInc"], y_test, alpha=0.2, label="Test observations")
for degree, model in models.items():
    plt.plot(x_line, model.predict(x_line), label=f"Degree {degree}")
plt.xlabel("Median income")
plt.ylabel("Median house value")
plt.ylim(0, 5.5)
plt.legend()
plt.show()


## 6. Interpret bias and variance

A low-degree model may underfit (high bias). Very high degrees can fit training data closely but may generalize poorly (high variance). Use the test metrics to support your conclusion.